# 🏆 Modelo Campeão: Stacking Ensemble com Erro Relativo (%) por Aviário

> **Documentação & Validação do Modelo Mais Promissor**  
> **Status:** Modelo em Produção / Recorde Histórico  
> **Métricas Alcançadas:** MAE = 92.11 g | RMSE = 130.86 g | R² = 0.5814 (~58.14%)  

---

## 📌 Contextualização & Alinhamento Estratégico

A predição acurada do peso corporal das aves na janela de abate (**42 a 60 dias de idade**) é essencial para o planejamento do frigorífico, otimização do escalonamento de apanha e minimização de perdas operacionais.

A solução estratégica para contornar falhas na entrega de ração e otimizar o planejamento de abates está estruturada sobre **três pilares fundamentais**:
1. **Comunicação Eficiente (Plataforma Centralizada):** Integração dos dados de pesagem amostral semanal de campo, relatórios de lote e dados do abatedouro em um único fluxo de dados centralizado.
2. **Processos Otimizados (Redesenho de Fluxo e Confirmação de Pedidos):** Redesenho da janela de confirmação de pedidos de ração e escalonamento de apanha de lote com base nas projeções de ganho de peso diário (GPD) e acurácia preditiva dos aviários.
3. **Tecnologia Habilitadora (TMS, Sensores de Nível nos Silos):** Uso de sistemas de gerenciamento de transporte (TMS) sincronizados às previsões de peso por lote e sensores IoT instalados nos silos de ração para evitar desabastecimento e estresse nutricional.



In [ ]:
# Configuração do Ambiente e Importação de Bibliotecas
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import joblib
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Definir caminho raiz do projeto
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

print('Ambiente carregado com sucesso!')



## 🔬 1. Modelagem Matemática Biológica: Curva de Gompertz & ARIMA

O crescimento das aves de corte segue uma curva sigmoidal biológica representada pela equação de Gompertz:
$$W(t) = A \cdot \exp(-b \cdot \exp(-k \cdot t))$$

Além da curva de Gompertz, utilizamos uma série temporal ARIMA(1,1,1) ajustada nos pesos médios diários para capturar variações dinâmicas de curto prazo.



In [ ]:
# Definição da Curva de Gompertz e Plotagem da Linha de Base Biológica
def gompertz_func(t, A=6260.16, b=4.7378, k=0.0449):
    return A * np.exp(-b * np.exp(-k * t))

idades = np.arange(1, 61)
pesos_gompertz = gompertz_func(idades)

plt.figure(figsize=(9, 5))
plt.plot(idades, pesos_gompertz / 1000.0, color='darkgreen', linewidth=2.5, label='Curva de Gompertz Calibrada')
plt.title('Curva Teórica de Crescimento Gompertz (1 a 60 Dias)', fontsize=13, fontweight='bold')
plt.xlabel('Idade (dias)', fontsize=11)
plt.ylabel('Peso Estimado (kg)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()



## 📈 2. Engenharia de Atributos: Erro Relativo (%) por Aviário & Dimensão Longitudinal

Para superar o viés de cada propriedade, calculamos a variação percentual relativa histórica de cada aviário em relação ao comportamento Gompertz:
$$\text{Erro Relativo}(\%) = \frac{W_{\text{observado}} - W_{\text{Gompertz}}}{W_{\text{Gompertz}}} \times 100$$

Esta feature permite que o modelo aprenda a eficiência tecnológica de cada aviário (climatizado vs convencional, isolamento, etc.).



In [ ]:
# Execução do Pipeline do Modelo Campeão
from src.models.aviary_relative_error_model import run_aviary_relative_error_experiment

# Executa o experimento de treino e avaliação em validação cruzada (GroupKFold)
run_aviary_relative_error_experiment()



## 📊 3. Desempenho Comparativo dos Modelos

Abaixo apresentamos a evolução das métricas ao longo das 5 fases do projeto:

| Fase / Iteração | MAE (g) | RMSE (g) | $R^2$ | Inovação Aplicada |
|---|:---:|:---:|:---:|---|
| **1. Modelo Estático Inicial** | 118,80 | 160,74 | 0,3684 | Baseline com atributos estáticos do lote |
| **2. Modelo Longitudinal** | 96,92 | 137,79 | 0,5359 | Pesagens aos 21d, 28d, 35d e GPD |
| **3. Modelo Tri-Híbrido** | 96,15 | 137,16 | 0,5401 | Fusão: Gompertz + ARIMA + Stacking ML |
| **4. Dimensão Delta (g) Aviário** | 92,57 | 131,54 | 0,5770 | Mapeamento do viés absoluto fixo em gramas |
| **5. Erro Relativo (%) Aviário** | **92,11** | **130,86** | **0,5814** | 🏆 **Recorde Histórico ($R^2 = 58,14\%$)** |



In [ ]:
# Visualização dos Gráficos Gerados pelo Modelo Campeão
plots_dir = os.path.join(project_root, 'plots')

plot_files = [
    'importancia_features_erro_relativo_aviario.png',
    'simulacoes_predito_vs_real_correcao_aviario.png',
    'comparativo_modelos_avancados.png'
]

for p in plot_files:
    full_p = os.path.join(plots_dir, p)
    if os.path.exists(full_p):
        print(f'Exibindo: {p}')
        display(Image(filename=full_p))
    else:
        print(f'Arquivo de imagem não encontrado: {full_p}')



## 🎯 4. Conclusões e Próximos Passos (Meta Sub-80g)

1. O **Modelo de Erro Relativo (%) por Aviário** reduziu o erro médio absoluto para **92,11 g** ($R^2 = 0,5814$), consolidando-se como o modelo oficial para o abatedouro.
2. A integração com os **3 Pilares** (Comunicação Centralizada, Processos Otimizados e Tecnologia TMS/Silos) garantirá maior previsibilidade de abates e eficiência na distribuição de ração.
3. Para as próximas iterações, a adição de telemetria de consumo de água e estresse térmico (ITU) visa atingir a meta sub-80g.

